# Multiagent systems

In [1]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()

True

In [2]:
model_client = OpenAIChatCompletionClient(model='gpt-4o')

# Sigle Agent approach 

In [ ]:
from autogen_agentchat.agents import AssistantAgent

story_agent = AssistantAgent(
    name= 'story_agent',
    model_client=model_client,
    system_message='you are a creative writer like director puri jagannadh in telugu.'
)

In [5]:
from autogen_agentchat.messages import TextMessage

async def test_simple_agent():
    task = TextMessage(
    content = 'generate a short story about an orphan boy in hyderabad in 50 words',
    source='user'
    )
    result = await story_agent.run(task=task)
    print(result.messages[-1].content)

await test_simple_agent()

In Hyderabad's bustling streets, orphan Ravi dreams big while selling garlands. Befriending Rani, a loyal stray, he discovers a mysterious map. Embarking on an adventurous quest, Ravi uncovers hidden city secrets. Inspired, he becomes a celebrated storyteller, transforming his life's challenges into tales of hope and wonder.


# Multiagent team approach

In [7]:
plot_agent = AssistantAgent(
    name='plot_agent',
    model_client=model_client,
    system_message='you create engaging plots for stories like ss rajamouli'
)

charecter_agent = AssistantAgent(
    name='charecter_agent',
    model_client=model_client,
    system_message='you develop charecters arcs like telugu director sukumar'
)

ending_agent = AssistantAgent(
    name='ending_agent',
    model_client=model_client,
    system_message='you write bindblowing endings for stories like christopher nolan'
)

In [10]:
from autogen_agentchat.teams import RoundRobinGroupChat
team = RoundRobinGroupChat(
    participants=[plot_agent, charecter_agent, ending_agent],
    max_turns=3
)

In [12]:
async def test_team():
    task = TextMessage(
        content='wrire a short story about orphan boy in hyderabad, keep it in under 200 words',
        source='user'
    )

    result = await team.run(task=task)
    for each_agent_message in result.messages:
        print(f'{each_agent_message.source}:{each_agent_message.content}')

await test_team()

user:wrire a short story about orphan boy in hyderabad, keep it in under 200 words
plot_agent:In the bustling lanes of Hyderabad, a spirited orphan named Arjun lived in the shadows of the ancient Charminar. Despite the city’s vibrant vitality, a deep sense of longing filled his young heart. Every morning, Arjun awoke with the sun’s first light, working at a small tea stall to earn his keep.

One day, while serving tea, he stumbled upon an old, leather-bound book abandoned on a rickety chair. Dusty and worn, its spine was embossed with the words **"Dreams of Nizams."** Opening it revealed a hidden map, seemingly depicting a secret passage beneath the city that led to a forgotten treasure of the Nizams.

Driven by curiosity, Arjun embarked on a covert journey. Each night, beneath the twinkling stars and the echoing calls of muezzins, he deciphered the ancient script and navigated through hidden tunnels and forgotten chambers.

In a forgotten vault adorned with jewels and artifacts, Arjun

In [ ]:
# #Single-Agent vs. Multi-Agent Comparison

# Aspect	            Single Agent	                    Multi-Agent Team
# Creativity	  Limited to one viewpoint	       Diverse ideas from each agent
# Depth	      Basic story elements	       Detailed plot, characters, ending
# Flexibility	 Stuck to one style	            Adaptable with specialized roles
# Setup Effort	Simple, one agent	      More agents, team setup


### single agent as team and RoundRobinGroupChat

In [14]:
from autogen_agentchat.teams import RoundRobinGroupChat

team2 = RoundRobinGroupChat(
    participants= [plot_agent],
    max_turns=2
)

from autogen_agentchat.base import TaskResult

# When running inside a script, use a async main function and call it from `asyncio.run(...)`.
await team2.reset()  # Reset the team for a new task.
async for message in team2.run_stream(task="Write a short story about the polictics in hyderabad in 50 words."):  # type: ignore
    if isinstance(message, TaskResult):
        print("Stop Reason:", message.stop_reason)
    else:
        print(message)

source='user' models_usage=None metadata={} content='Write a short story about the polictics in hyderabad in 50 words.' type='TextMessage'
source='plot_agent' models_usage=RequestUsage(prompt_tokens=40, completion_tokens=72) metadata={} content='In the bustling heart of Hyderabad, rival factions clashed, vying for control. Charismatic leader Arjun promised change, while cunning minister Rhea thrived in shadows. Amidst ancient bazaars and futuristic skyscrapers, alliances shifted, loyalties tested. In the end, a surprising coalition emerged, uniting the city, weaving tradition with modernity.' type='TextMessage'
source='plot_agent' models_usage=RequestUsage(prompt_tokens=116, completion_tokens=72) metadata={} content='In the bustling heart of Hyderabad, rival factions clashed, vying for control. Charismatic leader Arjun promised change, while cunning minister Rhea thrived in shadows. Amidst ancient bazaars and futuristic skyscrapers, alliances shifted, loyalties tested. In the end, a su

In [17]:
print(message)

messages=[TextMessage(source='user', models_usage=None, metadata={}, content='Write a short story about the polictics in hyderabad in 50 words.', type='TextMessage'), TextMessage(source='plot_agent', models_usage=RequestUsage(prompt_tokens=40, completion_tokens=72), metadata={}, content='In the bustling heart of Hyderabad, rival factions clashed, vying for control. Charismatic leader Arjun promised change, while cunning minister Rhea thrived in shadows. Amidst ancient bazaars and futuristic skyscrapers, alliances shifted, loyalties tested. In the end, a surprising coalition emerged, uniting the city, weaving tradition with modernity.', type='TextMessage'), TextMessage(source='plot_agent', models_usage=RequestUsage(prompt_tokens=116, completion_tokens=72), metadata={}, content='In the bustling heart of Hyderabad, rival factions clashed, vying for control. Charismatic leader Arjun promised change, while cunning minister Rhea thrived in shadows. Amidst ancient bazaars and futuristic skysc